# Task 2: carrying over Task 4's wins
Task 4 gained from (1) cosine similarity to the user's reads **near the row's time** and
(2) **EASE** item-item weights. Task 2 rows have no timestamp, so (1) uses each candidate's
own median read time as its query time: if the user read the book, they likely read it when
everyone else did. The two candidates get different query times.

Bar (plan_t2.md): keep a change only if it wins on every split, and aim for ≥ +0.003, since
small validation gains have shrunk by about two-thirds on public.

EASE on Task 2's 51k items runs in float32 (~10-11 GB peak, several minutes per fit).

In [1]:
%load_ext autoreload
%autoreload 2

import time

import numpy as np
import pandas as pd
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

from assignment_1.modules.models import ItemSimilarity, LocalSimilarity, binary_cosine, ease_similarity
from assignment_1.modules.tasks.task2 import task2_pair_features
from assignment_1.modules.util import load_csv, make_pair_validation

In [2]:
DATA_DIR = "../data"  # notebook runs from notebooks/

train = load_csv("tasks_1_2_train.csv", DATA_DIR)
tr, pairs = make_pair_validation(train, seed=0)
y = (pairs["which"] == pairs["item1"]).astype(int).to_numpy()
users = pairs["user"]
SEEDS = (0, 1, 2, 3)
halves = {s: np.random.default_rng(s).random(len(pairs)) < 0.5 for s in SEEDS}
print(f"{len(tr):,} tr rows | {len(pairs):,} pairs")

1,675,479 tr rows | 186,164 pairs


## Evaluation
Train on one half, score the other, for four random halves (seed 0 is the split in
`task2.ipynb`). Three combiners: `lr` (shipped: plain LogisticRegression), `lr_std`
(standardised first: new features are on very different scales), `gb`.
`wins` = splits (of 4) where the shipped `lr` beats the baseline.

In [3]:
COMBINERS = {
    "lr": lambda: LogisticRegression(max_iter=1000),
    "lr_std": lambda: make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000)),
    "gb": lambda: HistGradientBoostingClassifier(random_state=0),
}


def evaluate(X: np.ndarray) -> dict[str, np.ndarray]:
    """combiner -> accuracy per split."""
    return {name: np.array([(make().fit(X[h], y[h]).predict(X[~h]) == y[~h]).mean()
                            for h in halves.values()])
            for name, make in COMBINERS.items()}


results, raw = {}, {}


def record(name: str, F: np.ndarray | None) -> dict:
    X = X0 if F is None else np.hstack([X0, F.reshape(len(pairs), -1)])
    acc = evaluate(X)
    raw[name] = acc
    r = {f"{c}": round(float(a.mean()), 4) for c, a in acc.items()}
    if name != "baseline":
        base = raw["baseline"]
        for c in acc:
            r[f"Δ{c}"] = round(float((acc[c] - base[c]).mean()), 4)
        r["wins"] = int((acc["lr"] > base["lr"]).sum())
        r["Δgb_vs_base_lr"] = round(float((acc["gb"] - base["lr"]).mean()), 4)
    results[name] = r
    print(f"{name:<34} +{0 if F is None else F.reshape(len(pairs), -1).shape[1]:>2} cols  {r}")
    return r


def pair_diff(f) -> np.ndarray:
    """f(users, items) -> candidate features; returns f(item1) − f(item2)."""
    return np.asarray(f(users, pairs["item1"]), dtype=float) - \
        np.asarray(f(users, pairs["item2"]), dtype=float)

## Baseline: shipped features
Expect `lr` ≈ 0.9383 on seed 0 (`raw["baseline"]["lr"][0]`).

In [4]:
t0 = time.time()
X0 = np.asarray(task2_pair_features(tr, pairs), dtype=float)
print(f"baseline features {X0.shape} in {time.time() - t0:.0f}s")
record("baseline", None)
print("per split:", {c: a.round(4) for c, a in raw["baseline"].items()})

baseline features (186164, 11) in 151s
baseline                           + 0 cols  {'lr': 0.9378, 'lr_std': 0.9377, 'gb': 0.9377}
per split: {'lr': array([0.9383, 0.9382, 0.9373, 0.9372]), 'lr_std': array([0.9383, 0.9381, 0.9373, 0.9372]), 'gb': array([0.938 , 0.9377, 0.9376, 0.9375])}


## 1. EASE, global (sum / max / mean over the user's history)
Task 4's best λ was 25 (17k items). Task 2 has 51k items with fewer readers each, so the grid
leans low. Each fit is one float32 inversion of a 51k × 51k matrix.

In [5]:
ease_sims, F_ease = {}, {}
for lam in (10.0, 25.0, 100.0):
    t0 = time.time()
    sim = ItemSimilarity(ease_similarity(lam, keep=200, dtype=np.float32), prefix="ease").fit(tr)
    ease_sims[lam] = sim
    F_ease[lam] = pair_diff(lambda u, i: sim.features(u, i).to_numpy())
    print(f"λ={lam:g}: {time.time() - t0:.0f}s")
    record(f"EASE global λ={lam:g}", F_ease[lam])

λ=10: 213s
EASE global λ=10                   + 3 cols  {'lr': 0.939, 'lr_std': 0.9389, 'gb': 0.9399, 'Δlr': 0.0012, 'Δlr_std': 0.0012, 'Δgb': 0.0022, 'wins': 4, 'Δgb_vs_base_lr': 0.0021}
λ=25: 215s
EASE global λ=25                   + 3 cols  {'lr': 0.9389, 'lr_std': 0.9388, 'gb': 0.9398, 'Δlr': 0.0011, 'Δlr_std': 0.0011, 'Δgb': 0.0021, 'wins': 4, 'Δgb_vs_base_lr': 0.0021}
λ=100: 213s
EASE global λ=100                  + 3 cols  {'lr': 0.9385, 'lr_std': 0.9386, 'gb': 0.9399, 'Δlr': 0.0007, 'Δlr_std': 0.0009, 'Δgb': 0.0022, 'wins': 4, 'Δgb_vs_base_lr': 0.0022}


## 2. Time-local cosine at the candidate's typical read time
Query time per candidate = its median (or 25th-percentile) training timestamp.
Then the same `LocalSimilarity` as Task 4: the user's 20 reads nearest that time.

In [6]:
item_t = {
    "median": tr.groupby("item")["timestamp"].median(),
    "q25": tr.groupby("item")["timestamp"].quantile(0.25),
}
cos_sim = ItemSimilarity(binary_cosine).fit(tr)
local_cos = LocalSimilarity(binary_cosine).fit(tr, sim=cos_sim)


def at_item_time(loc: LocalSimilarity, tmap: pd.Series):
    return lambda u, i: loc.features(u, i, i.map(tmap).fillna(0)).to_numpy()


F_loc = {}
for name, tmap in item_t.items():
    t0 = time.time()
    F_loc[name] = pair_diff(at_item_time(local_cos, tmap))
    print(f"{name}: {time.time() - t0:.0f}s")
    record(f"local cos @ item {name}", F_loc[name])

median: 36s
local cos @ item median            +13 cols  {'lr': 0.9382, 'lr_std': 0.9383, 'gb': 0.9384, 'Δlr': 0.0004, 'Δlr_std': 0.0005, 'Δgb': 0.0007, 'wins': 4, 'Δgb_vs_base_lr': 0.0006}
q25: 37s
local cos @ item q25               +13 cols  {'lr': 0.9383, 'lr_std': 0.9385, 'gb': 0.9384, 'Δlr': 0.0005, 'Δlr_std': 0.0008, 'Δgb': 0.0007, 'wins': 4, 'Δgb_vs_base_lr': 0.0006}


Sensitivity of the best query time: neighbourhood size.

In [7]:
BEST_T = "median"  # set from the table above
for n_side in (5, 20):
    loc = LocalSimilarity(binary_cosine, n_side=n_side).fit(tr, sim=cos_sim)
    record(f"local cos @ {BEST_T} n={n_side}", pair_diff(at_item_time(loc, item_t[BEST_T])))

local cos @ median n=5             +13 cols  {'lr': 0.9381, 'lr_std': 0.9382, 'gb': 0.9383, 'Δlr': 0.0003, 'Δlr_std': 0.0005, 'Δgb': 0.0006, 'wins': 3, 'Δgb_vs_base_lr': 0.0005}
local cos @ median n=20            +13 cols  {'lr': 0.9382, 'lr_std': 0.9383, 'gb': 0.9385, 'Δlr': 0.0004, 'Δlr_std': 0.0006, 'Δgb': 0.0009, 'wins': 4, 'Δgb_vs_base_lr': 0.0008}


## 3. Time-local EASE at the candidate's read time (set `BEST_LAM` from section 1)

In [8]:
BEST_LAM = 25.0
local_ease = LocalSimilarity(prefix="eloc").fit(tr, sim=ease_sims[BEST_LAM])
F_eloc = pair_diff(at_item_time(local_ease, item_t[BEST_T]))
record(f"local EASE λ={BEST_LAM:g} @ {BEST_T}", F_eloc)

local EASE λ=25 @ median           +13 cols  {'lr': 0.9379, 'lr_std': 0.9382, 'gb': 0.9384, 'Δlr': 0.0001, 'Δlr_std': 0.0004, 'Δgb': 0.0007, 'wins': 3, 'Δgb_vs_base_lr': 0.0006}


{'lr': 0.9379,
 'lr_std': 0.9382,
 'gb': 0.9384,
 'Δlr': 0.0001,
 'Δlr_std': 0.0004,
 'Δgb': 0.0007,
 'wins': 3,
 'Δgb_vs_base_lr': 0.0006}

## 4. Combine

In [9]:
record(f"EASE λ={BEST_LAM:g} + local cos", np.hstack([F_ease[BEST_LAM], F_loc[BEST_T]]))
record(f"EASE λ={BEST_LAM:g} + local cos + local EASE",
       np.hstack([F_ease[BEST_LAM], F_loc[BEST_T], F_eloc]))

EASE λ=25 + local cos              +16 cols  {'lr': 0.9393, 'lr_std': 0.9392, 'gb': 0.9404, 'Δlr': 0.0015, 'Δlr_std': 0.0015, 'Δgb': 0.0027, 'wins': 4, 'Δgb_vs_base_lr': 0.0027}
EASE λ=25 + local cos + local EASE +29 cols  {'lr': 0.9393, 'lr_std': 0.9392, 'gb': 0.9406, 'Δlr': 0.0015, 'Δlr_std': 0.0015, 'Δgb': 0.0029, 'wins': 4, 'Δgb_vs_base_lr': 0.0028}


{'lr': 0.9393,
 'lr_std': 0.9392,
 'gb': 0.9406,
 'Δlr': 0.0015,
 'Δlr_std': 0.0015,
 'Δgb': 0.0029,
 'wins': 4,
 'Δgb_vs_base_lr': 0.0028}

In [10]:
pd.DataFrame(results).T

,lr,lr_std,gb,Δlr,Δlr_std,Δgb,wins,Δgb_vs_base_lr
baseline,0.9378,0.9377,0.9377,NaN,NaN,NaN,NaN,NaN
EASE global λ=10,0.9390,0.9389,0.9399,0.0012,0.0012,0.0022,4.0,0.0021
EASE global λ=25,0.9389,0.9388,0.9398,0.0011,0.0011,0.0021,4.0,0.0021
EASE global λ=100,0.9385,0.9386,0.9399,0.0007,0.0009,0.0022,4.0,0.0022
local cos @ item median,0.9382,0.9383,0.9384,0.0004,0.0005,0.0007,4.0,0.0006
local cos @ item q25,0.9383,0.9385,0.9384,0.0005,0.0008,0.0007,4.0,0.0006
local cos @ median n=5,0.9381,0.9382,0.9383,0.0003,0.0005,0.0006,3.0,0.0005
local cos @ median n=20,0.9382,0.9383,0.9385,0.0004,0.0006,0.0009,4.0,0.0008
local EASE λ=25 @ median,0.9379,0.9382,0.9384,0.0001,0.0004,0.0007,3.0,0.0006
EASE λ=25 + local cos,0.9393,0.9392,0.9404,0.0015,0.0015,0.0027,4.0,0.0027
